# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/widadfatimakhan/flyrank-internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane 4 — CTR / Engagement Opportunity Scoring.**

Six weeks produced a validated score. A score is not a product. This notebook turns it into a
**ranked list of actions a person can act on**, with the reason attached, the limits stated, and the
things that must never be automated written down.

**The design idea from this week's session, and the whole notebook rests on it:**

> **The frozen rule supplies the REASON. The model supplies the ORDER.**

They do different jobs. The ML-07 rule can say *why* a page is on the list — it is below its
position peers by a stated amount. The ML-08/09 model cannot say why; it only sorts. Fusing them
gives an editor a queue that is both ordered and explainable.

| Card asks for | Lives in |
|---|---|
| Ranked actions + reason codes | §1 |
| Intended use and limits | §2 |
| Human review + the no-go list | §3 |
| Monitoring / retrain triggers | §4 |
| Exports for the paper | §5 |

**This is a decision-support plan, not a production system.** Nothing here is deployed to servers.
Every policy below is labelled with where it is enforced: **ASSERTED** in this notebook,
**UPSTREAM** in the data contract, or **PROPOSED** and not implemented.

## 0. Setup — the deployment frame

The queue is built at the **2026-04-01 decision moment**: March features, scored by the model
trained on February→March. That is the honest deployment posture from ML-09 — the model was fitted
before April existed.

April outcomes are loaded too, but **only to evaluate** the queue after the fact. They are never an
input to it.

In [14]:
%pip -q install --upgrade duckdb

import os, getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
print("token loaded:", bool(HF_TOKEN))

token loaded: True


In [15]:
import duckdb, pandas as pd, numpy as np, sklearn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 60)
SEED = 42
print(f"duckdb {duckdb.__version__} | sklearn {sklearn.__version__} | pandas {pd.__version__}")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL  = "hf://datasets/FlyRank/internship-warehouse"
DIMC = f"read_parquet('{REL}/dim_content.parquet')"
def month_rel(m): return f"read_parquet('{REL}/fact_content_daily_performance/month={m}/*.parquet')"

MIN_IMPRESSIONS, MIN_ACTIVE_DAYS, MIN_POSITION = 500, 5, 1.0     # ML-04 eligibility
GAP_MIN, CLICKS_MIN, OUT_MIN_IMPRESSIONS       = 0.10, 10, 100   # ML-07 thresholds
DECISION_MOMENT = pd.Timestamp("2026-04-01")
REVIEW_BUDGET   = 50          # pages one editor can actually open in a month
HALVES = {"2026-02": ("2026-02-01", "2026-02-15"),
          "2026-03": ("2026-03-04", "2026-03-18"),
          "2026-04": ("2026-04-03", "2026-04-17")}

duckdb 1.5.5 | sklearn 1.6.1 | pandas 2.2.3


In [16]:
def page_month(month):
    prev_from, last_from = HALVES[month]
    df = con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)                                                AS impressions,
               SUM(gsc_clicks)                                                     AS clicks,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END)  AS active_days,
               MAX(gsc_impressions)                                                AS top_day_impressions,
               SUM(gsc_sum_position) FILTER (WHERE gsc_impressions > 0
                                               AND gsc_avg_position IS NOT NULL)   AS pos_num,
               SUM(gsc_impressions)  FILTER (WHERE gsc_impressions > 0
                                               AND gsc_avg_position IS NOT NULL)   AS pos_den,
               STDDEV_SAMP(gsc_avg_position) FILTER (WHERE gsc_impressions > 0
                                               AND gsc_avg_position IS NOT NULL)   AS position_volatility,
               SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '{last_from}') AS imp_last_half,
               SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '{prev_from}'
                                              AND report_date <  DATE '{last_from}') AS imp_prev_half
        FROM {month_rel(month)}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
        ORDER BY client_hash_id, content_hash_id
    """).df()
    df["avg_position"] = df.pos_num / df.pos_den.replace(0, np.nan) + 1   # zero-based fix, ML-04
    df["ctr_pp"]       = 100 * df.clicks / df.impressions.replace(0, np.nan)
    return df

def add_gap(df, tag):
    df = df.copy()
    df["tier"] = df.avg_position.apply(
        lambda p: "top_3" if p <= 3 else "page_1" if p <= 10 else "striking" if p <= 20
        else "page_3_5" if p <= 50 else "deep")
    tc, ti = df.groupby("tier").clicks.transform("sum"), df.groupby("tier").impressions.transform("sum")
    df[f"{tag}_peer_pp"]       = (100 * (tc - df.clicks) /
                                  (ti - df.impressions)).replace([np.inf, -np.inf], np.nan)
    df[f"{tag}_gap_pp"]        = df[f"{tag}_peer_pp"] - df.ctr_pp
    df[f"{tag}_missed_clicks"] = df.impressions * df[f"{tag}_gap_pp"] / 100
    return df

# last_optimized_date is pulled deliberately -- see the correction note in section 3.
dimc = con.sql(f"""SELECT content_hash_id, word_count, content_type,
                          content_created_date, last_optimized_date
                   FROM {DIMC}""").df()
months = {m: page_month(m) for m in ["2026-02", "2026-03", "2026-04"]}
for m, d in months.items():
    print(f"{m}: {len(d):>7,} pages")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-02: 153,559 pages
2026-03: 176,738 pages
2026-04: 194,760 pages


In [17]:
SHAPE = ["log_impressions", "active_days", "position_volatility", "top_day_share",
         "momentum_log", "impressions_share_of_client", "log_word_count", "content_meta_missing"]
PRIOR = ["gap_pp", "ctr_pp_feat", "log_clicks"]
FEATS = SHAPE + PRIOR

def build_frame(feature_month, outcome_month):
    f = months[feature_month]
    f = f[(f.impressions >= MIN_IMPRESSIONS) & (f.active_days >= MIN_ACTIVE_DAYS) &
          (f.avg_position >= MIN_POSITION)].copy()
    n_eligible = len(f)
    f = add_gap(f, "f")
    f["log_impressions"] = np.log1p(f.impressions)
    f["top_day_share"]   = f.top_day_impressions / f.impressions
    f["momentum_log"]    = np.log((f.imp_last_half.fillna(0) + 1) / (f.imp_prev_half.fillna(0) + 1))
    f["impressions_share_of_client"] = (f.impressions /
                                        f.groupby("client_hash_id").impressions.transform("sum"))
    f = f.merge(dimc, on="content_hash_id", how="left", validate="many_to_one")
    wc = pd.to_numeric(f.word_count, errors="coerce").astype("float64")
    f["content_meta_missing"] = wc.isna().astype(int)
    f["log_word_count"] = np.log1p(wc.fillna(wc.groupby(f.client_hash_id).transform("median"))
                                     .fillna(wc.median()))
    f["gap_pp"], f["ctr_pp_feat"], f["log_clicks"] = f.f_gap_pp, f.ctr_pp, np.log1p(f.clicks)

    o = months[outcome_month]
    o = o[(o.impressions >= OUT_MIN_IMPRESSIONS) & (o.avg_position >= MIN_POSITION)].copy()
    o = add_gap(o, "o")
    o["label"] = ((o.o_gap_pp >= GAP_MIN) & (o.o_missed_clicks >= CLICKS_MIN)).astype(int)
    out = f.merge(o[["content_hash_id", "label", "o_gap_pp", "impressions", "avg_position"]]
                    .rename(columns={"impressions": "out_impressions", "avg_position": "out_position"}),
                  on="content_hash_id", how="inner", validate="one_to_one")
    out = out.dropna(subset=FEATS + ["label"]).copy()
    out.attrs["n_eligible"] = n_eligible
    return out

train_df  = build_frame("2026-02", "2026-03")     # what the model was fitted on
queue_df  = build_frame("2026-03", "2026-04")     # the 2026-04-01 queue (labels for evaluation only)
print(f"TRAIN  Feb->Mar : {len(train_df):>7,} rows | base rate {train_df.label.mean():.3f}")
print(f"QUEUE  Mar->Apr : {len(queue_df):>7,} rows | base rate {queue_df.label.mean():.3f} "
      f"(labels used ONLY to evaluate, never to build)")
assert len(queue_df) == 60942, "the ML-08/09 frame did not reproduce -- stop and investigate"
print("reproduction check: PASS (matches the committed ML-08 and ML-09 receipts)")

TRAIN  Feb->Mar :  45,931 rows | base rate 0.119
QUEUE  Mar->Apr :  60,942 rows | base rate 0.095 (labels used ONLY to evaluate, never to build)
reproduction check: PASS (matches the committed ML-08 and ML-09 receipts)


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### The fusion

| component | job | why it is the right tool for that job |
|---|---|---|
| **ML-07 frozen rule** | the **reason** | it is arithmetic a person can check: this page's CTR is X% against peers at Y%, a gap of Z percentage points worth N clicks |
| **ML-08/09 model** | the **order** | it ranks better at the head of the queue (precision@50 0.96 vs 0.90), but it cannot say why — a probability is not an explanation |

An editor opening item #3 sees a **score** telling them it is worth their time and a **sentence**
telling them what to look at. Neither alone is enough.

### The four actions

| action | fires when | what the editor does |
|---|---|---|
| `SNIPPET_REVIEW` | rule flags it, traffic is steady, position is not top-2 | read the title and meta description against the page and the query intent |
| `VERIFY_THEN_REVIEW` | flagged, but traffic is spiky or swung hard | check for a tracking or campaign artefact **first** — a broken tag looks exactly like a broken snippet |
| `MONITOR_ONLY` | impressions rising durably | watch it. A page gaining traffic is not a page to rewrite |
| `INVESTIGATE_QUIET_RISK` | model ranks it high but the rule returns **no flag** | a person must look. **A high score with no stated reason is a question, not an instruction** |

That last one is the session's point made concrete: *"Just because the model score is high does not
mean there is a clear reason. The model never tells us why on its own."*

Everything else is `NO_ACTION` — and the queue is allowed to say that about most of the portfolio.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.ensemble import RandomForestClassifier

def rf(): return RandomForestClassifier(n_estimators=60, max_depth=10, min_samples_leaf=40,
                                        random_state=SEED, n_jobs=1)

# The deployed model: fitted BEFORE April existed, applied unchanged. ML-09's time-forward posture.
model = rf().fit(train_df[FEATS].values, train_df.label.values)
q = queue_df.copy()
q["model_score"] = model.predict_proba(q[FEATS].values)[:, 1]

# --- the REASON comes from the frozen rule, not the model --------------------
q["rule_flag"] = ((q.f_gap_pp >= GAP_MIN) & (q.f_missed_clicks >= CLICKS_MIN))
q["reason_code"] = np.where(q.rule_flag, "CTR_BELOW_POSITION_PEERS", "NO_FLAG")
q["reason_text"] = np.where(
    q.rule_flag,
    ("CTR " + q.ctr_pp.round(2).astype(str) + "% vs peers at similar position "
     + q.f_peer_pp.round(2).astype(str) + "% (gap " + q.f_gap_pp.round(2).astype(str)
     + "pp, about " + q.f_missed_clicks.round(0).astype(int).astype(str) + " clicks/month)"),
    "no rule reason: within the peer range for its position")

print(f"rule reason present : {int(q.rule_flag.sum()):,} pages "
      f"({q.rule_flag.mean():.1%} of the eligible portfolio)")
print(f"no rule reason      : {int((~q.rule_flag).sum()):,} pages")

rule reason present : 6,014 pages (9.9% of the eligible portfolio)
no rule reason      : 54,928 pages


In [19]:
# --- the ACTION: gates applied in order, first match wins --------------------
SPIKE_SHARE, BIG_SWING, DURABLE_RISE, TOP_BAND = 0.30, np.log(20), 0.35, 2.0

def assign_action(r):
    if r.momentum_log >= BIG_SWING:                      # >20x swing between half-months
        return "VERIFY_THEN_REVIEW"
    if r.momentum_log >= DURABLE_RISE and not r.rule_flag:
        return "MONITOR_ONLY"
    if r.rule_flag and r.top_day_share > SPIKE_SHARE:
        return "VERIFY_THEN_REVIEW"
    if r.rule_flag and r.avg_position <= TOP_BAND:
        return "INVESTIGATE_QUIET_RISK"                  # ML-06: peer fairness fails at 1-2
    if r.rule_flag:
        return "SNIPPET_REVIEW"
    if r.model_score >= q.model_score.quantile(0.99):
        return "INVESTIGATE_QUIET_RISK"                  # high score, no stated reason
    return "NO_ACTION"

q["action"] = q.apply(assign_action, axis=1)

# --- the ORDER: model score, ties broken by a seeded jitter (ML-09 policy) ----
jit = np.random.default_rng(SEED).random(len(q)) * 1e-12
q = q.iloc[np.lexsort((jit, -q.model_score.values))].reset_index(drop=True)
q["rank"] = np.arange(1, len(q) + 1)

actionable = q[q.action != "NO_ACTION"].copy()
actionable["action_rank"] = np.arange(1, len(actionable) + 1)
print("ACTION MIX across the eligible portfolio")
print(q.action.value_counts().to_string())
print(f"\nactionable items: {len(actionable):,} | review budget this month: {REVIEW_BUDGET}")
print(f"the top {REVIEW_BUDGET} of the actionable queue is what an editor actually sees.\n")

top = actionable.head(REVIEW_BUDGET)
print(f"top-{REVIEW_BUDGET} action mix: {dict(top.action.value_counts())}")
display(top.head(8)[["action_rank", "action", "reason_code", "model_score", "impressions",
                     "avg_position", "ctr_pp", "f_peer_pp", "f_gap_pp", "f_missed_clicks"]].round(3))
print("\nA worked queue line, as an editor would read it:")
r0 = top.iloc[0]
print(f"  #{int(r0.action_rank)}  {r0.action}  [model score {r0.model_score:.2f}]")
print(f"      {r0.reason_text}")

ACTION MIX across the eligible portfolio
action
NO_ACTION                 39225
MONITOR_ONLY              14025
SNIPPET_REVIEW             5835
VERIFY_THEN_REVIEW         1795
INVESTIGATE_QUIET_RISK       62

actionable items: 21,717 | review budget this month: 50
the top 50 of the actionable queue is what an editor actually sees.

top-50 action mix: {'SNIPPET_REVIEW': np.int64(48), 'INVESTIGATE_QUIET_RISK': np.int64(2)}


,action_rank,action,reason_code,model_score,impressions,avg_position,ctr_pp,f_peer_pp,f_gap_pp,f_missed_clicks
0,1,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,0.971,17875.0,5.439,0.090,0.340,0.251,44.834
1,2,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,0.969,17487.0,3.858,0.091,0.340,0.249,43.514
2,3,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,0.968,17265.0,2.415,0.081,0.317,0.236,40.723
3,4,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,0.968,13184.0,7.876,0.083,0.340,0.257,33.868
4,5,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,0.967,19674.0,5.851,0.081,0.340,0.259,50.958
5,6,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,0.964,26507.0,5.267,0.068,0.340,0.272,72.216
6,7,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,0.964,19217.0,3.239,0.088,0.340,0.252,48.402
7,8,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,0.964,21719.0,6.446,0.055,0.340,0.285,61.919



A worked queue line, as an editor would read it:
  #1  SNIPPET_REVIEW  [model score 0.97]
      CTR 0.09% vs peers at similar position 0.34% (gap 0.25pp, about 45 clicks/month)


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Who uses this, for what

**A FlyRank content editor**, once a month, to decide **which pages to open first**. The budget is
about 50 pages. With a budget that small, *the order is the product* — precision at the head of the
queue is the only thing that matters, which is why precision@50 was fixed as the metric in Week 4,
before any model was trained.

### What the queue is, in one sentence

**Observed** on 60,942 pages across 34 clients, a Random Forest trained on February→March and
applied unchanged to the March→April window **measured** precision@50 of 0.96 against the frozen
Week-4 rule at 0.90 and a base rate of 0.095, with the K=200 margin holding a bootstrap interval
that excludes zero. The improvement is **directional**, not uniform. The queue is
**decision-support** for what to open first — it does not diagnose what is wrong with a page, and
nothing here shows that editing one changes its outcome.

### Where it stops being valid

| limit | why it matters here |
|---|---|
| **One month of forward evidence** | ML-09 tested exactly one train-then-deploy transition. That is thin temporal evidence, and it is a limitation of the claim, not a footnote |
| **The label is size-influenced by construction** | the ≥10-missed-clicks gate takes the base rate from 0.4% among small pages to 47.1% among large ones. The queue is partly a large-page detector, and that is measured, not suspected |
| **The label cannot separate "fixed" from "died"** | two of ML-08's three most confident errors were pages whose April traffic fell by more than half. Their gap stopped clearing the threshold because demand left |
| **Survivor selection** | pages with no readable April data are excluded — about 1.5% of the eligible set. They may be the ones that failed hardest |
| **Peer fairness fails at positions 1–2** | ML-06 showed every client tested converts worse on its own 1–2 pages than its own 3–5 pages. Those pages are routed to `INVESTIGATE_QUIET_RISK`, never straight to a rewrite |
| **Clients differ by ~10×** | per-client pooled CTR ran 0.115% to 1.159%, and per-client model AUC ran 0.600 to 0.990. It does not work equally well for everyone |
| **No calibration** | the column is called `model_score`, **not** confidence or probability. Calibration was never evaluated, so a 0.8 does not mean an 80% chance |

### What it is not

Not a production system. Not a ranking-factor study — **a predictive feature is not a search
ranking factor**. Not evidence that refreshing content causes recovery; that needs a controlled
test nobody has run.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- Cost and value, with the arithmetic visible ----------------------------
def precision_at_k(scores, y, k):
    j = np.random.default_rng(SEED).random(len(scores)) * 1e-12
    return float(np.mean(np.asarray(y)[np.lexsort((j, -np.asarray(scores)))[:k]]))

y = q.label.values
p_model = precision_at_k(q.model_score.values, y, REVIEW_BUDGET)
p_rule  = precision_at_k(q.f_missed_clicks.values, y, REVIEW_BUDGET)
base    = y.mean()
found_model, found_rule, found_rand = p_model*REVIEW_BUDGET, p_rule*REVIEW_BUDGET, base*REVIEW_BUDGET

print("COST AND VALUE -- derived arithmetic, with its assumptions on show\n")
print(f"in a {REVIEW_BUDGET}-page review batch, of pages whose gap persisted into April:")
print(f"  random ordering  ~{found_rand:5.1f} pages")
print(f"  frozen rule      ~{found_rule:5.0f} pages")
print(f"  model order      ~{found_model:5.0f} pages   (+{found_model - found_rule:.0f} vs the rule)")
print(f"\ncost side: {REVIEW_BUDGET} reviews x ~20 minutes = ~{REVIEW_BUDGET*20/60:.0f} editor-hours/month")
print(f"clicks at stake in the top {REVIEW_BUDGET} (rule arithmetic): "
      f"{top.f_missed_clicks.sum():,.0f}/month")
print("\nASSUMPTIONS, stated: this is a descriptive scenario from one evaluated window, not expected")
print("future impact. It assumes a page could reach its position peers' CTR -- no experiment here")
print("tests that -- and it makes no claim about traffic recovery, which needs a controlled test.")
print("\nWould a plain rule create most of the value with less risk? Largely yes: the rule finds")
print(f"{found_rule:.0f} of the {found_model:.0f} the model finds. The model's contribution is the")
print(f"last {found_model - found_rule:.0f} pages, and it costs a fitted artefact that must be")
print("monitored and retrained. Section 4 makes that trade a policy rather than a preference.")

COST AND VALUE -- derived arithmetic, with its assumptions on show

in a 50-page review batch, of pages whose gap persisted into April:
  random ordering  ~  4.7 pages
  frozen rule      ~   45 pages
  model order      ~   48 pages   (+3 vs the rule)

cost side: 50 reviews x ~20 minutes = ~17 editor-hours/month
clicks at stake in the top 50 (rule arithmetic): 2,870/month

ASSUMPTIONS, stated: this is a descriptive scenario from one evaluated window, not expected
future impact. It assumes a page could reach its position peers' CTR -- no experiment here
tests that -- and it makes no claim about traffic recovery, which needs a controlled test.

Would a plain rule create most of the value with less risk? Largely yes: the rule finds
45 of the 48 the model finds. The model's contribution is the
last 3 pages, and it costs a fitted artefact that must be
monitored and retrained. Section 4 makes that trade a policy rather than a preference.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Four questions, asked before a single word is rewritten:

1. **Did a special event or season cause the change?** A seasonal peak is not a snippet problem.
2. **Is the page still about what people are searching for?** Intent drifts; a rewrite aimed at the
   old intent makes things worse.
3. **Is the client changing things right now?** A migration or redesign in flight explains most
   anomalies better than any model does.
4. **Does the reason sentence actually match what I see on the page?** If the rule says the gap is
   0.4pp and the page is a login screen nobody clicks by design, the queue is wrong and the editor
   overrules it.

**The queue tells you what to look at. It never authorises a rewrite.** *If this playbook can be
used without anyone thinking, it was built wrong.*

### The no-go list — and where each policy is actually enforced

| # | policy | status |
|---|---|---|
| 1 | **Thin-history pages are never queued.** Fewer than 5 active days makes momentum meaningless — a new page can look like a 20× riser because it started at nothing | **UPSTREAM** — data contract eligibility, ML-04 |
| 2 | **Below-floor pages are never queued.** Under 500 impressions, one click moves CTR by more than the flagging threshold | **UPSTREAM** — data contract eligibility, ML-04 |
| 3 | **Durable risers are monitor-only, never rewritten** | **ASSERTED** in §1 and re-checked below |
| 4 | **Swings above 20× go to verification first** — that is a tracking or campaign artefact until proven otherwise | **ASSERTED** in §1 and re-checked below |
| 5 | **Recently optimised pages are excluded (90-day cooldown)** | **ASSERTED** below — see the correction |
| 6 | **Position 1–2 pages never get a straight rewrite instruction** — they are routed to investigation, because ML-06 showed the peer comparison is unfair exactly there | **ASSERTED** in §1 and re-checked below |

### A correction to ML-07

ML-07's receipts recorded: *"cooldown: NOT IMPLEMENTABLE — release has no optimisation history."*

**That was wrong.** `dim_content` carries `last_optimized_date`, and this week's schema read found
it. The cooldown is implemented below, and the pages it removes are counted. I state the correction
here rather than quietly fixing it, because the earlier claim is in a committed receipt.

### What must never be automated

- **No automatic rewriting.** The queue proposes; a person disposes.
- **No automatic client-facing reporting.** A page can top this queue and be performing exactly as
  intended — a navigational page, a page whose query Google answers directly.
- **No treating `model_score` as a probability.** It is uncalibrated.
- **No acting on `INVESTIGATE_QUIET_RISK` as if it were a review instruction.** A high score with no
  rule reason means *nobody knows why yet*.



In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- The gates, asserted in code --------------------------------------------
q["last_opt"] = pd.to_datetime(q.last_optimized_date, errors="coerce")
q["days_since_optimised"] = (DECISION_MOMENT - q.last_opt).dt.days
COOLDOWN_DAYS = 90
in_cooldown = q.days_since_optimised.notna() & (q.days_since_optimised < COOLDOWN_DAYS)

print("NO-GO GATES, checked against the queue that would ship\n")
blocked = q[in_cooldown & (q.action == "SNIPPET_REVIEW")]
q.loc[in_cooldown & (q.action == "SNIPPET_REVIEW"), "action"] = "NO_ACTION"
q.loc[in_cooldown, "reason_code"] = np.where(q.loc[in_cooldown, "rule_flag"],
                                             "SUPPRESSED_COOLDOWN", q.loc[in_cooldown, "reason_code"])
print(f"[5 ASSERTED] cooldown: {int(in_cooldown.sum()):,} pages optimised within {COOLDOWN_DAYS} days "
      f"of {DECISION_MOMENT.date()}; {len(blocked):,} of them were headed for SNIPPET_REVIEW and are "
      f"now suppressed")
print(f"             pages with NO last_optimized_date at all: {int(q.last_opt.isna().sum()):,} "
      f"({q.last_opt.isna().mean():.1%}) -- these pass the gate by default, which is a known hole")

for label, mask, expect in [
    ("[3 ASSERTED] durable risers never get SNIPPET_REVIEW",
     (q.momentum_log >= DURABLE_RISE) & (~q.rule_flag) & (q.action == "SNIPPET_REVIEW"), 0),
    ("[4 ASSERTED] >20x swings go to verification, not review",
     (q.momentum_log >= BIG_SWING) & (q.action == "SNIPPET_REVIEW"), 0),
    ("[6 ASSERTED] position 1-2 never gets a straight rewrite",
     (q.avg_position <= TOP_BAND) & (q.action == "SNIPPET_REVIEW"), 0)]:
    n = int(mask.sum())
    print(f"{label}: {n} violations {'PASS' if n == expect else 'FAIL'}")
    assert n == expect, label

print(f"\n[1-2 UPSTREAM] every page here already cleared the ML-04 contract: >= {MIN_IMPRESSIONS} "
      f"impressions, >= {MIN_ACTIVE_DAYS} active days, position >= {MIN_POSITION}")

# rebuild the actionable queue after the cooldown suppression
actionable = q[q.action != "NO_ACTION"].copy().reset_index(drop=True)
actionable["action_rank"] = np.arange(1, len(actionable) + 1)
top = actionable.head(REVIEW_BUDGET)
print(f"\nfinal action mix after all gates:")
print(q.action.value_counts().to_string())

print(f"\nThe queue that actually ships, after every gate:")
print(f"  top-{REVIEW_BUDGET} action mix: "
      f"{ {k: int(v) for k, v in top.action.value_counts().items()} }")
print(f"  the section-1 table was the PRE-GATE ordering; {len(blocked):,} pages shown there as "
      f"SNIPPET_REVIEW are suppressed by cooldown and do not reach an editor.")
display(top.head(8)[["action_rank", "action", "reason_code", "model_score", "impressions",
                     "avg_position", "ctr_pp", "f_gap_pp", "f_missed_clicks",
                     "days_since_optimised"]].round(3))
print(f"\nNote on the NaN column: {top.days_since_optimised.isna().mean():.0%} of the shipping "
      f"top-{REVIEW_BUDGET} has no recorded optimisation date, so the cooldown could not evaluate "
      f"them and they passed by default. That is the 48.2% hole named above, showing up exactly "
      f"where it matters most -- at the head of the queue a person will actually work.")

NO-GO GATES, checked against the queue that would ship

[5 ASSERTED] cooldown: 31,575 pages optimised within 90 days of 2026-04-01; 4,383 of them were headed for SNIPPET_REVIEW and are now suppressed
             pages with NO last_optimized_date at all: 29,367 (48.2%) -- these pass the gate by default, which is a known hole
[3 ASSERTED] durable risers never get SNIPPET_REVIEW: 0 violations PASS
[4 ASSERTED] >20x swings go to verification, not review: 0 violations PASS
[6 ASSERTED] position 1-2 never gets a straight rewrite: 0 violations PASS

[1-2 UPSTREAM] every page here already cleared the ML-04 contract: >= 500 impressions, >= 5 active days, position >= 1.0

final action mix after all gates:
action
NO_ACTION                 43608
MONITOR_ONLY              14025
VERIFY_THEN_REVIEW         1795
SNIPPET_REVIEW             1452
INVESTIGATE_QUIET_RISK       62

The queue that actually ships, after every gate:
  top-50 action mix: {'SNIPPET_REVIEW': 48, 'INVESTIGATE_QUIET_RISK': 2}
  th

,action_rank,action,reason_code,model_score,impressions,avg_position,ctr_pp,f_gap_pp,f_missed_clicks,days_since_optimised
0,1,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,0.971,17875.0,5.439,0.090,0.251,44.834,NaN
1,2,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,0.969,17487.0,3.858,0.091,0.249,43.514,NaN
2,3,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,0.968,13184.0,7.876,0.083,0.257,33.868,NaN
3,4,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,0.964,19217.0,3.239,0.088,0.252,48.402,NaN
4,5,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,0.964,21719.0,6.446,0.055,0.285,61.919,NaN
5,6,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,0.964,19024.0,5.834,0.095,0.246,46.745,NaN
6,7,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,0.964,19187.0,3.305,0.073,0.267,51.300,NaN
7,8,SNIPPET_REVIEW,CTR_BELOW_POSITION_PEERS,0.964,15472.0,4.663,0.084,0.256,39.656,NaN



Note on the NaN column: 96% of the shipping top-50 has no recorded optimisation date, so the cooldown could not evaluate them and they passed by default. That is the 48.2% hole named above, showing up exactly where it matters most -- at the head of the queue a person will actually work.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Monitoring has **two clocks**, and confusing them is how teams end up believing they had warning
when they did not.

| clock | when it can run | what it can see |
|---|---|---|
| **Pre-release** | before the queue ships, no labels needed | population size, schema, feature distribution drift |
| **Post-label** | only after the outcome month completes | the new base rate, and how the model did against the rule |

**The honest limit:** post-label checks can change the *next* queue. They cannot change what has
already been sent to an editor. So a shift is always discovered one cycle late.

### The proposed thresholds — policy, not a shipped system

| trigger | threshold | clock | action |
|---|---|---|---|
| population change | >±20% **against the same calendar month a year earlier** | pre-release | hold the queue, inspect the contract |
| feature drift (PSI) | any feature PSI > 0.25 | pre-release | flag for review before release |
| base-rate shift | >±10 percentage points | post-label | pause the next release pending review |
| model below rule | 2 consecutive labeled months | post-label | **fall back to the frozen rule** |

*Corrected after running it: a month-over-month comparison fires on calendar length alone —
February has 28 days, March has 31. See the verdict below.*

That last one is the cost/value question turned into a rule: **complexity must keep proving it is
useful.** If the model stops beating a rule a person can read, the rule wins.

**PSI** (Population Stability Index) compares a feature's distribution between two periods. The
convention: under 0.10 is no meaningful shift, 0.10–0.25 moderate, above 0.25 major. It is
implemented in this notebook only — it is not wired into anything.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- Pre-release clock: population, schema, drift ---------------------------
def psi(expected, actual, bins=10):
    cuts = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    if len(cuts) < 3: return 0.0
    e = np.histogram(expected, bins=cuts)[0] / len(expected)
    a = np.histogram(actual,   bins=cuts)[0] / len(actual)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

pop_change = (len(queue_df) - len(train_df)) / len(train_df)
print(f"[PRE-RELEASE] population: {len(train_df):,} -> {len(queue_df):,} ({pop_change:+.1%}) "
      f"{'TRIGGER' if abs(pop_change) > 0.20 else 'ok'} (threshold +/-20%)")
missing = [c for c in FEATS if c not in queue_df.columns]
print(f"[PRE-RELEASE] schema: {len(missing)} expected feature(s) missing {'FAIL' if missing else 'PASS'}")

drift = pd.DataFrame([{"feature": f, "psi": round(psi(train_df[f].values, queue_df[f].values), 4)}
                      for f in FEATS]).sort_values("psi", ascending=False)
drift["verdict"] = np.where(drift.psi > 0.25, "MAJOR", np.where(drift.psi > 0.10, "moderate", "stable"))
display(drift)
trig = drift[drift.psi > 0.25]
print(f"[PRE-RELEASE] feature drift: {len(trig)} feature(s) above 0.25 "
      f"{'-> TRIGGER: hold and inspect' if len(trig) else '-> no trigger'}")
print("Note: this compares the training window to the scoring window, which is the comparison a "
      "pre-release check can actually make -- it needs no labels.")

[PRE-RELEASE] population: 45,931 -> 60,942 (+32.7%) TRIGGER (threshold +/-20%)
[PRE-RELEASE] schema: 0 expected feature(s) missing PASS


,feature,psi,verdict
1,active_days,2.1128,MAJOR
6,log_word_count,0.4895,MAJOR
4,momentum_log,0.2001,moderate
2,position_volatility,0.1369,moderate
3,top_day_share,0.0541,stable
8,gap_pp,0.0277,stable
0,log_impressions,0.0176,stable
9,ctr_pp_feat,0.0090,stable
10,log_clicks,0.0074,stable
5,impressions_share_of_client,0.0072,stable


[PRE-RELEASE] feature drift: 2 feature(s) above 0.25 -> TRIGGER: hold and inspect
Note: this compares the training window to the scoring window, which is the comparison a pre-release check can actually make -- it needs no labels.


In [23]:
# --- Post-label clock: what we learn only after the outcome month closes -----
train_base, queue_base = train_df.label.mean(), queue_df.label.mean()
shift = queue_base - train_base
print(f"[POST-LABEL] base rate {train_base:.3f} (Feb->Mar) -> {queue_base:.3f} (Mar->Apr) "
      f"({shift:+.3f})  {'TRIGGER: pause next release' if abs(shift) > 0.10 else 'ok'} "
      f"(threshold +/-10pp)")
print(f"[POST-LABEL] model precision@{REVIEW_BUDGET} {p_model:.3f} vs frozen rule {p_rule:.3f} "
      f"-> {'model ahead' if p_model > p_rule else 'RULE AHEAD: strike 1 of 2 toward fallback'}")
print(f"[POST-LABEL] both against a base rate of {base:.3f}")

pre_fired  = bool((abs(pop_change) > 0.20) or (drift.psi > 0.25).any())
post_fired = bool(abs(shift) > 0.10 or (p_model <= p_rule))
print(f"\nCOMBINED VERDICT -- pre-release: {'TRIGGERED' if pre_fired else 'clear'} | "
      f"post-label: {'TRIGGERED' if post_fired else 'clear'}")

print("\nAnd the pre-release triggers are a finding about MY POLICY, not about the model.")
print(f"  February has 28 days; March has 31. Median active_days "
      f"{train_df.active_days.median():.0f} -> {queue_df.active_days.median():.0f}.")
print(f"  active_days therefore CANNOT exceed 28 in training and can reach 31 in scoring, which is")
print(f"  most of its PSI of {drift.set_index('feature').loc['active_days','psi']:.2f}. The same")
print(f"  calendar explains the {pop_change:+.1%} population change: fewer pages clear a "
      f"{MIN_IMPRESSIONS}-impression")
print("  floor in a shorter month, so the eligible set grows going into a longer one.")
print("\nOBSERVED: as written, these thresholds would fire on every February->March transition for")
print("reasons unrelated to model health -- and a monitor that cries wolf on the calendar gets")
print("ignored by month three. CORRECTION TO THE POLICY, before it is ever used: normalise")
print("active_days by days-in-month, and compare population against the same calendar month a year")
print("earlier rather than the month before. The base-rate and model-vs-rule checks are unaffected,")
print("because both are rates.")
print("\nThe limit that stands: I have ONE labeled transition. A base-rate alarm calibrated on one")
print("observation is a guess with a number attached. June 2026 stays sealed and would be the next")
print("honest test of whether any of these thresholds is set near right.")

[POST-LABEL] base rate 0.119 (Feb->Mar) -> 0.095 (Mar->Apr) (-0.024)  ok (threshold +/-10pp)
[POST-LABEL] model precision@50 0.960 vs frozen rule 0.900 -> model ahead
[POST-LABEL] both against a base rate of 0.095

COMBINED VERDICT -- pre-release: TRIGGERED | post-label: clear

And the pre-release triggers are a finding about MY POLICY, not about the model.
  February has 28 days; March has 31. Median active_days 28 -> 31.
  active_days therefore CANNOT exceed 28 in training and can reach 31 in scoring, which is
  most of its PSI of 2.11. The same
  calendar explains the +32.7% population change: fewer pages clear a 500-impression
  floor in a shorter month, so the eligible set grows going into a longer one.

OBSERVED: as written, these thresholds would fire on every February->March transition for
reasons unrelated to model health -- and a monitor that cries wolf on the calendar gets
ignored by month three. CORRECTION TO THE POLICY, before it is ever used: normalise
active_days by days

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Three artefacts, and the split between them is deliberate:

| file | destination | committed? |
|---|---|---|
| the ranked queue | `work/outputs/content_action_playbook_queue.csv` | **no** — data file, blocked by the CI leak-guard, regenerated on every run |
| precision@K figure | `work/figures/precision_at_k.png` | **yes** — the paper reuses it |
| action-mix figure | `work/figures/action_mix.png` | **yes** |
| metrics receipt | `work/outputs/ml10_playbook_receipts.json` | **yes** — the receipt the paper's numbers trace back to |

**On the figures.** Lines are directly labelled and use different dash styles, so colour is not the
only way to tell them apart. Each caption says what to notice, which claim it supports, and — for
the precision@K chart — what it does **not** prove, printed with the figure rather than buried in a
limitations section.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, json
os.makedirs("work/outputs", exist_ok=True); os.makedirs("work/figures", exist_ok=True)

# --- figure 1: precision@K, model vs frozen rule vs base rate ----------------
Ks = [10, 25, 50, 75, 100, 150, 200, 300, 500]
pm = [precision_at_k(q.model_score.values, y, k) for k in Ks]
pr = [precision_at_k(q.f_missed_clicks.values, y, k) for k in Ks]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(Ks, pm, marker="o", linestyle="-",  color="#1f4e79", label="model order")
ax.plot(Ks, pr, marker="s", linestyle="--", color="#a6511f", label="frozen rule")
ax.axhline(base, linestyle=":", color="#555555", label=f"base rate ({base:.3f})")
ax.axvspan(0, REVIEW_BUDGET, alpha=0.10, color="#1f4e79")
ax.annotate("review budget lives here", xy=(REVIEW_BUDGET, max(pm)*0.55),
            xytext=(REVIEW_BUDGET*1.6, max(pm)*0.45), fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#333333"))
ax.set_xlabel("K (pages reviewed)"); ax.set_ylabel("precision@K (higher is better)")
ax.set_title("Queue precision by review depth, Mar-2026 features -> Apr-2026 outcomes")
ax.legend(frameon=False); ax.grid(alpha=0.25)
fig.tight_layout(); fig.savefig("work/figures/precision_at_k.png", dpi=150); plt.close(fig)

CAPTION_1 = (
 f"Figure 1. Precision@K for the model order and the frozen Week-4 rule on {len(q):,} pages across "
 f"{q.client_hash_id.nunique()} clients, features from March 2026 and observed outcomes from April "
 "2026; the model was fitted on February->March and applied unchanged. Higher is better. The dotted "
 f"line is the base rate ({base:.3f}) -- what a random queue returns. The shaded band is the "
 f"{REVIEW_BUDGET}-page review budget, which is where the comparison matters. WHAT THIS DOES NOT "
 "PROVE: it does not show that any feature is a search ranking factor, and it does not show that "
 "editing a flagged page causes its gap to close. Predictive, not causal.")
print(CAPTION_1)

Figure 1. Precision@K for the model order and the frozen Week-4 rule on 60,942 pages across 34 clients, features from March 2026 and observed outcomes from April 2026; the model was fitted on February->March and applied unchanged. Higher is better. The dotted line is the base rate (0.095) -- what a random queue returns. The shaded band is the 50-page review budget, which is where the comparison matters. WHAT THIS DOES NOT PROVE: it does not show that any feature is a search ranking factor, and it does not show that editing a flagged page causes its gap to close. Predictive, not causal.


In [25]:
# --- figure 2: action mix ----------------------------------------------------
mix = q.action.value_counts().reindex(
    ["SNIPPET_REVIEW", "VERIFY_THEN_REVIEW", "INVESTIGATE_QUIET_RISK", "MONITOR_ONLY", "NO_ACTION"]
).dropna()
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(mix.index[::-1], mix.values[::-1], color="#1f4e79")
for b, v in zip(bars, mix.values[::-1]):
    ax.text(v + max(mix.values)*0.01, b.get_y() + b.get_height()/2,
            f"{v:,} ({v/len(q):.1%})", va="center", fontsize=9)
ax.set_xlabel("pages"); ax.set_xlim(0, max(mix.values)*1.22)
ax.set_title("What the playbook proposes across the eligible portfolio")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout(); fig.savefig("work/figures/action_mix.png", dpi=150); plt.close(fig)

CAPTION_2 = (
 f"Figure 2. Action mix across {len(q):,} eligible pages at the 2026-04-01 decision moment. "
 "Reason codes come from the frozen Week-4 rule; the order within each action comes from the model. "
 "Most of the portfolio is NO_ACTION by design -- a queue that cannot say 'nothing this month' is "
 "unreadable. INVESTIGATE_QUIET_RISK covers pages the model ranks highly with no rule reason "
 "attached, and pages at positions 1-2 where the peer comparison was shown to be unfair.")
print(CAPTION_2)
print(f"\naction mix:\n{mix.to_string()}")

Figure 2. Action mix across 60,942 eligible pages at the 2026-04-01 decision moment. Reason codes come from the frozen Week-4 rule; the order within each action comes from the model. Most of the portfolio is NO_ACTION by design -- a queue that cannot say 'nothing this month' is unreadable. INVESTIGATE_QUIET_RISK covers pages the model ranks highly with no rule reason attached, and pages at positions 1-2 where the peer comparison was shown to be unfair.

action mix:
action
SNIPPET_REVIEW             1452
VERIFY_THEN_REVIEW         1795
INVESTIGATE_QUIET_RISK       62
MONITOR_ONLY              14025
NO_ACTION                 43608


In [26]:
# --- export the queue + receipts --------------------------------------------
COLS = ["rank", "action_rank", "client_hash_id", "content_hash_id", "action", "reason_code",
        "reason_text", "model_score", "impressions", "clicks", "ctr_pp", "f_peer_pp", "f_gap_pp",
        "f_missed_clicks", "avg_position", "active_days", "top_day_share", "momentum_log",
        "days_since_optimised"]
export = actionable[COLS].copy()
export.to_csv("work/outputs/content_action_playbook_queue.csv", index=False)
print(f"wrote work/outputs/content_action_playbook_queue.csv -- {len(export):,} actionable rows "
      f"x {len(COLS)} columns (stays out of git; regenerated every run)")

receipts = {
    "assignment": "ML-10 - Content Action Playbook",
    "decision_moment": str(DECISION_MOMENT.date()),
    "design": "frozen ML-07 rule supplies the reason; ML-08/09 model supplies the order",
    "model": {"trained_on": "2026-02 features -> 2026-03 outcomes",
              "applied_to": "2026-03 features -> 2026-04 outcomes", "refit": False, "seed": SEED},
    "population": {"eligible_pages": int(len(q)), "clients": int(q.client_hash_id.nunique()),
                   "base_rate": round(float(base), 4)},
    "queue": {"review_budget": REVIEW_BUDGET,
              "action_mix": {k: int(v) for k, v in q.action.value_counts().items()},
              "actionable_rows": int(len(actionable))},
    "performance": {"precision_at_50_model": round(p_model, 3),
                    "precision_at_50_rule": round(p_rule, 3),
                    "base_rate": round(float(base), 4),
                    "extra_persistent_pages_per_50": round(found_model - found_rule, 1)},
    "no_go_policies": [
        {"policy": "thin-history pages never queued", "status": "UPSTREAM (ML-04 contract)"},
        {"policy": "below-floor pages never queued", "status": "UPSTREAM (ML-04 contract)"},
        {"policy": "durable risers monitor-only", "status": "ASSERTED"},
        {"policy": ">20x swings verify first", "status": "ASSERTED"},
        {"policy": f"{COOLDOWN_DAYS}-day cooldown on recently optimised pages", "status": "ASSERTED",
         "note": "corrects ML-07, which recorded this as NOT IMPLEMENTABLE; "
                 "dim_content.last_optimized_date exists",
         "suppressed": int(len(blocked)),
         "pages_with_no_optimisation_date": int(q.last_opt.isna().sum())},
        {"policy": "position 1-2 never gets a straight rewrite", "status": "ASSERTED"},
    ],
    "monitoring": {"pre_release": {"population_change": round(float(pop_change), 4),
                                   "max_feature_psi": float(drift.psi.max()),
                                   "features_above_0_25": int((drift.psi > 0.25).sum())},
                   "post_label": {"base_rate_train": round(float(train_base), 4),
                                  "base_rate_deploy": round(float(queue_base), 4),
                                  "shift": round(float(shift), 4),
                                  "model_ahead_of_rule": bool(p_model > p_rule)},
                   "status": "PROPOSED policy, implemented in this notebook only"},
    "exports": ["work/outputs/content_action_playbook_queue.csv (not committed)",
                "work/figures/precision_at_k.png", "work/figures/action_mix.png"],
    "figure_captions": {"figure_1": CAPTION_1, "figure_2": CAPTION_2},
    "claim_discipline": "observed, measured, directional, decision-support; no causal claim",
    "open_checks": ["only one time-forward window evaluated",
                    "no calibration -- model_score is not a probability",
                    "no causal evidence: no page was edited as an experiment",
                    "June 2026 remains sealed"],
}
with open("work/outputs/ml10_playbook_receipts.json", "w") as f:
    json.dump(receipts, f, indent=2, default=float)
print("saved -> work/outputs/ml10_playbook_receipts.json")
print("saved -> work/figures/precision_at_k.png, work/figures/action_mix.png")

wrote work/outputs/content_action_playbook_queue.csv -- 17,334 actionable rows x 19 columns (stays out of git; regenerated every run)
saved -> work/outputs/ml10_playbook_receipts.json
saved -> work/figures/precision_at_k.png, work/figures/action_mix.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### What next week's paper takes from here

1. **The ranked recommendations section** is §1 and §3 — actions, reason codes, guardrails, and the
   measure for each.
2. **Two figures with captions already written**, each stating what it shows and what it does not
   prove.
3. **The cost/value box** with its arithmetic and assumptions visible, not a promise of impact.
4. **The limitations** in §2, each tied to a specific claim rather than offered as a general
   disclaimer.

### Still open, carried forward

- Only **one** time-forward window has been evaluated. More windows would strengthen every claim
  here, and this belongs in the paper's limitations.
- **No causal evidence.** Whether refreshing a page causes recovery is untested, and only a
  controlled experiment would show it.
- **June 2026 stays sealed** — it is the next honest test of whether the monitoring thresholds in
  §4 are set anywhere near right.